# El problema del viajante
Tienes ocho ciudades. Busca una ruta corta que pase por todas y regrese al punto de partida. Una ruta es una tupla de índices, como `(0, 2, 1, …)`. Mantendremos la ciudad 0 al principio.

Aplicaremos la escalada a una ruta inicial. En cada paso, compararemos rutas que difieran en un único intercambio y elegiremos la más corta si mejora la actual.

In [ ]:
from itertools import combinations, permutations
from math import dist, exp
from random import Random

rng = Random(1)
ciutats = [(rng.randrange(100), rng.randrange(100)) for _ in range(8)]
inicial = tuple(range(len(ciutats)))

def cost(ruta):
    circuit = ruta + (ruta[0],)
    return sum(dist(ciutats[a], ciutats[b]) for a, b in zip(circuit, circuit[1:]))

def intercanvia(ruta, i, j):
    nova = list(ruta)
    nova[i], nova[j] = nova[j], nova[i]
    return tuple(nova)

def veines(ruta):
    for i, j in combinations(range(1, len(ruta)), 2):
        yield intercanvia(ruta, i, j)

## El primer paso de la escalada
Compara el coste de la ruta inicial con el del mejor vecino. Si el coste baja, harías este intercambio.

In [ ]:
millor_vei = min(veines(inicial), key=cost)
print(inicial, round(cost(inicial), 2))
print(millor_vei, round(cost(millor_vei), 2))

## Hill climbing
En cada paso intercambiamos las posiciones de dos ciudades. Probamos los vecinos y elegimos el mejor si reduce el coste. Si ningún intercambio mejora la ruta, hemos llegado a un mínimo local.

In [ ]:
def hill_climbing(ruta):
    while True:
        millor = min(veines(ruta), key=cost, default=ruta)
        if cost(millor) >= cost(ruta):
            return ruta
        ruta = millor

local = hill_climbing(inicial)
print(local, round(cost(local), 2))

## ¿Es la mejor ruta posible?
Cuando la escalada se detiene, ningún intercambio mejora la ruta. Para saber si hay un circuito más corto, comparamos las 7! rutas que empiezan en 0. Con ocho ciudades es viable. El número de rutas crece rápidamente al añadir ciudades.

In [ ]:
optima = min(((0,) + resta for resta in permutations(inicial[1:])), key=cost)
print(optima, round(cost(optima), 2))
print(round(cost(local) - cost(optima), 2))  # Diferencia de coste

## Variantes de la búsqueda local
Con **reinicios aleatorios** repetimos hill climbing desde rutas distintas y guardamos la mejor. Con **recocido simulado** probamos un intercambio aleatorio cada vez. Si empeora el coste en `delta`, lo aceptamos con probabilidad `exp(-delta / temperatura)`. Enfriamos en cada paso y guardamos la mejor ruta que hemos visto.

**Primer mejor** prueba los vecinos en orden aleatorio y elige el primero que mejora la ruta.

In [ ]:
def primer_millor(ruta, rng):
    while True:
        candidates = list(veines(ruta))
        rng.shuffle(candidates)
        for candidata in candidates:
            if cost(candidata) < cost(ruta):
                ruta = candidata
                break
        else:
            return ruta

def amb_reinicis(ruta, rng, intents=10):
    millor = hill_climbing(ruta)
    for _ in range(intents):
        nova = (ruta[0],) + tuple(rng.sample(ruta[1:], len(ruta) - 1))
        resultat = hill_climbing(nova)
        if cost(resultat) < cost(millor):
            millor = resultat
    return millor

def recuit(ruta, rng, passos=500, temperatura=30):
    millor = ruta
    for _ in range(passos):
        i, j = rng.sample(range(1, len(ruta)), 2)
        candidata = intercanvia(ruta, i, j)
        delta = cost(candidata) - cost(ruta)
        if delta <= 0 or rng.random() < exp(-delta / temperatura):
            ruta = candidata
            if cost(ruta) < cost(millor):
                millor = ruta
        temperatura *= 0.99
    return millor

In [ ]:
primer = primer_millor(inicial, Random(42))
reinicis = amb_reinicis(inicial, Random(42))
recuit_final = recuit(inicial, Random(42))
for nom, ruta in (("Inicial", inicial), ("Hill climbing", local),
                  ("Primer mejor", primer),
                  ("Reinicis / Reinicios", reinicis),
                  ("Recuit / Recocido", recuit_final), ("Òptim / Óptimo", optima)):
    print(f"{nom}: {cost(ruta):.2f}  {ruta}")

Cambia la semilla de los reinicios o la temperatura del recocido. ¿Llegas siempre al mismo coste que con la búsqueda exacta?

## Visualiza las rutas
Compara el circuito que da la escalada con el circuito óptimo. El dibujo incluye el último tramo, que vuelve a la ciudad 0.

In [ ]:
from IPython.display import HTML, display

def dibuixa_ruta(ruta, titol):
    punts = [(20 + 2*x, 220 - 2*y) for x, y in ciutats]
    circuit = ruta + (ruta[0],)
    linia = " ".join(f"{punts[i][0]},{punts[i][1]}" for i in circuit)
    svg = [f'<svg viewBox="0 0 240 260" width="240" role="img" aria-label="{titol}">',
           f'<polyline points="{linia}" fill="none" stroke="#2668a3" stroke-width="2"/>']
    for i, (x, y) in enumerate(punts):
        svg.append(f'<circle cx="{x}" cy="{y}" r="4" fill="#2668a3"/>')
        svg.append(f'<text x="{x + 5}" y="{y - 5}" fill="#444">{i}</text>')
    svg.append(f'<text x="16" y="248" fill="#444">{titol} · {cost(ruta):.2f}</text></svg>')
    return "".join(svg)

display(HTML(dibuixa_ruta(local, "Hill climbing") + dibuixa_ruta(optima, "Óptimo")))